In [88]:
import pandas as pd
import numpy as np
import psycopg2
import sqlalchemy as db
from sqlalchemy import create_engine
import yaml

In [89]:
with open('C:\\Users\\dylan\\OneDrive\\Documentos\\mensajeria\\config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['mensajeria']
    config_etl = config['ETL_PRO']

# Construct the database URL
url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
# Create the SQLAlchemy Engine
mensajeria = create_engine(url_co)
etl_conn = create_engine(url_etl)

In [90]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)
tabla_servicios = pd.read_sql_table('mensajeria_servicio', mensajeria)
 
tabla_estados=pd.read_sql_table('mensajeria_estadosservicio', mensajeria)
tabla_sede = pd.read_sql_table('clientes_usuarioaquitoy', mensajeria)

tabla_servicios.rename(columns={'id':'servicio_id'}, inplace=True)

tabla_servicios["key_trans_servicio"] = range(1,len(tabla_servicios)+1 )


tabla_servicios.drop(columns=['novedades', 'fecha_deseada', 'hora_deseada','nombre_recibe','telefono_recibe','hora_visto_por_mensajero',
                              'descripcion_pago','ida_y_regreso','tipo_pago_id','prioridad','visto_por_mensajero','multiples_origenes',
                              'descripcion_cancelado','descripcion_multiples_origenes','activo','asignar_mensajero','es_prueba',
                              'nombre_solicitante','descripcion','origen_id','destino_id','tipo_vehiculo_id','tipo_servicio_id'], inplace=True)



tabla_servicios = tabla_servicios.merge(tabla_sede[['cliente_id','user_id','sede_id']],
                                        left_on=['cliente_id','usuario_id'], right_on=['cliente_id','user_id'], how='left')

tabla_servicios['mensajero_id']=tabla_servicios['mensajero_id'].fillna(0).astype(int)
tabla_servicios['mensajero2_id']=tabla_servicios['mensajero2_id'].fillna(0).astype(int)
tabla_servicios['mensajero3_id']=tabla_servicios['mensajero3_id'].fillna(0).astype(int)

print(tabla_servicios['mensajero_id'].value_counts()) 
tabla_servicios.info()


mensajero_id
30    2439
29    1553
15    1514
25    1456
31    1352
16    1333
41    1329
42    1254
22    1252
28    1228
11    1101
27    1068
8     1059
18     920
3      917
44     849
32     732
34     727
0      727
45     686
38     622
4      604
36     562
24     558
12     436
48     396
5      185
23     179
47     164
40     137
49     129
19     127
33     120
46     112
83      94
43      91
17      87
21      78
39      73
7       68
37      65
13      30
9       13
1        2
2        1
84       1
Name: count, dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28430 entries, 0 to 28429
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   servicio_id         28430 non-null  int64         
 1   fecha_solicitud     28430 non-null  datetime64[ns]
 2   hora_solicitud      28430 non-null  object        
 3   cliente_id          28430 non-null  int64         
 4   mensa

In [91]:
print(len(tabla_servicios))  
tabla_estados_pivot_fecha = tabla_estados.pivot_table(
    index='servicio_id',
    columns='estado_id',
    values='fecha',
    aggfunc='first'      
).reset_index()
print(len(tabla_servicios))  

tabla_estados_pivot_hora = tabla_estados.pivot_table(
    index='servicio_id',
    columns='estado_id',
    values='hora',
    aggfunc='first'      
).reset_index()
trans_servicios = tabla_servicios.merge(tabla_estados_pivot_fecha, on='servicio_id', how='left')
print(len(tabla_servicios))  
trans_servicios.drop(columns=[3], inplace=True)
trans_servicios.rename(columns={1:'fecha_iniciado', 2:'fecha_mensajero_asignado', 4:'fecha_recogido_mensajero',
                                 5:'fecha_entregado',6:'fecha_finalizado_completo'}, inplace=True)
trans_servicios = trans_servicios.merge(tabla_estados_pivot_hora, on='servicio_id', how='left')
trans_servicios.drop(columns=[3], inplace=True)
trans_servicios.rename(columns={1:'hora_iniciado', 2:'hora_mensajero_asignado', 4:'hora_recogido_mensajero',
                                 5:'hora_entregado',6:'hora_finalizado_completo'}, inplace=True)
trans_servicios.info()


28430
28430
28430
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28430 entries, 0 to 28429
Data columns (total 23 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   servicio_id                28430 non-null  int64         
 1   fecha_solicitud            28430 non-null  datetime64[ns]
 2   hora_solicitud             28430 non-null  object        
 3   cliente_id                 28430 non-null  int64         
 4   mensajero_id               28430 non-null  int64         
 5   usuario_id                 28430 non-null  int64         
 6   ciudad_destino_id          28430 non-null  int64         
 7   ciudad_origen_id           28430 non-null  int64         
 8   mensajero2_id              28430 non-null  int64         
 9   mensajero3_id              28430 non-null  int64         
 10  key_trans_servicio         28430 non-null  int64         
 11  user_id                    3579 non-null   float6

In [92]:
trans_servicios['mensajero_id'].value_counts()

mensajero_id
30    2439
29    1553
15    1514
25    1456
31    1352
16    1333
41    1329
42    1254
22    1252
28    1228
11    1101
27    1068
8     1059
18     920
3      917
44     849
32     732
34     727
0      727
45     686
38     622
4      604
36     562
24     558
12     436
48     396
5      185
23     179
47     164
40     137
49     129
19     127
33     120
46     112
83      94
43      91
17      87
21      78
39      73
7       68
37      65
13      30
9       13
1        2
2        1
84       1
Name: count, dtype: int64

In [93]:
trans_servicios.to_sql('trans_servicio', etl_conn, if_exists='replace', index=False)



430